# 02 — Phase Kickback

**Trilha:** Pensamento espectral  
**Milestone:** 2 — Computação quântica como processamento espectral

---

## Pergunta

Como um circuito consegue transferir informação de fase de um operador para um registrador que pode ser medido?

Este é o notebook onde o circuito começa a "dizer algo" sobre o operador, não apenas aplicar portas. Phase kickback é o mecanismo central que habilita phase estimation — e, por extensão, grande parte dos algoritmos quânticos úteis.

## Modelo matemático mínimo

### O mecanismo

Considere um operador unitário $U$ com autovetor $|u\rangle$ e autovalor $e^{i\phi}$:
$$U|u\rangle = e^{i\phi}|u\rangle$$

Agora considere um registrador de controle preparado em superposição $\frac{|0\rangle + |1\rangle}{\sqrt{2}}$ e o alvo em $|u\rangle$.

O operador controlado $\text{ctrl-}U$ age como:
$$\text{ctrl-}U \left(\frac{|0\rangle + |1\rangle}{\sqrt{2}} \otimes |u\rangle\right) = \frac{|0\rangle \otimes |u\rangle + |1\rangle \otimes U|u\rangle}{\sqrt{2}}$$

$$= \frac{|0\rangle + e^{i\phi}|1\rangle}{\sqrt{2}} \otimes |u\rangle$$

A fase $e^{i\phi}$ **migrou** do registrador alvo para o registrador de controle. O alvo voltou ao estado original. O controle carrega agora a informação espectral de $U$.

### Por que isso é não-trivial

Classicamente, para extrair o autovalor de um operador precisamos resolver um sistema linear ou fazer diagonalização explícita. Aqui, a fase aparece diretamente no registrador de controle — e pode ser lida por interferência.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Verificação matricial do phase kickback ---

# Operador U com autovalor e^(iφ) sobre |u> = |1>
phi = np.pi / 3  # fase a ser extraída
U = np.diag([1.0, np.exp(1j * phi)])  # U|0> = |0>, U|1> = e^(iφ)|1>

# Autovetores e autovalores
eigvals, eigvecs = np.linalg.eig(U)
print(f'Autovalores de U: {eigvals}')
print(f'Fase do autovalor não-trivial: φ = {np.angle(eigvals[1]):.6f}')
print(f'Fase esperada: π/3 = {np.pi/3:.6f}')

In [ ]:
# --- Simulação do kickback com álgebra matricial ---

# Estado inicial: controle em |+>, alvo em |1> (autovetor de U)
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)  # controle
ket_1    = np.array([0, 1], dtype=complex)               # alvo (autovetor)

# Produto tensorial
state_in = np.kron(ket_plus, ket_1)
print('Estado inicial (controle ⊗ alvo):', state_in.round(4))

# Operador ctrl-U: age U no alvo quando controle é |1>
I = np.eye(2, dtype=complex)
P0 = np.outer([1, 0], [1, 0])  # projetor |0><0|
P1 = np.outer([0, 1], [0, 1])  # projetor |1><1|
ctrl_U = np.kron(P0, I) + np.kron(P1, U)

# Aplicar
state_out = ctrl_U @ state_in
print('Estado final (controle ⊗ alvo): ', state_out.round(4))

# Verificar: alvo deve voltar a |1>, controle deve ter a fase kickback
# Esperado: (|0> + e^(iφ)|1>) / sqrt(2)  ⊗  |1>
expected_ctrl = np.array([1, np.exp(1j * phi)], dtype=complex) / np.sqrt(2)
expected = np.kron(expected_ctrl, ket_1)
print('Estado esperado:               ', expected.round(4))
print(f'Diferença: {np.linalg.norm(state_out - expected):.2e}  (deve ser ~0)')

In [ ]:
# --- Visualização: fase no registrador de controle ---

# Após o kickback, o registrador de controle está em (|0> + e^(iφ)|1>) / sqrt(2)
# Medindo na base X ou Y, extraímos informação sobre φ

# Vamos variar φ e ver como a medição em X muda
phases = np.linspace(0, 2 * np.pi, 200)
expectation_X = []
expectation_Y = []

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

for ph in phases:
    ctrl = np.array([1, np.exp(1j * ph)], dtype=complex) / np.sqrt(2)
    expectation_X.append(np.real(ctrl.conj() @ X @ ctrl))
    expectation_Y.append(np.real(ctrl.conj() @ Y @ ctrl))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(phases / np.pi, expectation_X, label='⟨X⟩ = cos(φ)')
ax.plot(phases / np.pi, expectation_Y, label='⟨Y⟩ = sin(φ)', linestyle='--')
ax.set_xlabel('Fase φ (unidades de π)')
ax.set_ylabel('Valor esperado')
ax.set_title('Phase kickback: informação de fase no registrador de controle')
ax.legend()
ax.axhline(0, color='gray', alpha=0.3)
plt.tight_layout()
plt.show()

print('O controle encoda φ como rotação no plano X-Y da esfera de Bloch.')
print('Phase estimation usará interferência para ler φ com precisão arbitrária.')

In [ ]:
# --- O que acontece quando alvo NÃO é autovetor ---

# Alvo em superposição de autovetores de U
# U = diag(1, e^(iπ/3)) -> autovetores são |0> e |1>
# Alvo em |+> = (|0> + |1>) / sqrt(2)
ket_target_super = np.array([1, 1], dtype=complex) / np.sqrt(2)
state_in_super = np.kron(ket_plus, ket_target_super)

state_out_super = ctrl_U @ state_in_super
print('Estado saída (alvo em superposição):')
print(state_out_super.round(4))

# Verificar emaranhamento: estado não é mais produto tensorial
# Não há uma fase única: o controle e alvo ficam emaranhados
print()
print('Quando o alvo não é autovetor, controle e alvo ficam EMARANHADOS.')
print('Não há uma fase única sendo kickeada — cada componente espectral contribui.')
print('Phase estimation lida com isso via interferência no registrador de controle.')

## Conclusão

1. **Phase kickback é o mecanismo de transferência de informação espectral.** Quando o alvo está em um autovetor de $U$, a fase do autovalor migra para o controle sem perturbar o alvo.

2. **A informação de fase fica codificada no controle como rotação na esfera de Bloch.** Isso é legível via medição em X ou Y — ou via interferência (como faz QPE).

3. **O truque requer que o alvo seja (ou seja preparado como) autovetor de $U$.** Se o alvo está em superposição de autovetores, controle e alvo ficam emaranhados e a leitura não é direta. Phase estimation lida com isso, mas o custo aparece.

4. **Isso não é aceleração — é extração.** O circuito não resolve mais rápido; ele extrai informação espectral que seria inacessível por leitura direta do operador.

---

**Próximo:** como usar kickback repetidamente com potências de $U$ para estimar $\phi$ com precisão arbitrária — phase estimation completo.